In [ ]:
# Install the required packages:
# - langchain: core LangChain framework
# - langchain-openai: OpenAI model integration
!pip install -q langchain langchain-openai


In [ ]:
from google.colab import userdata

# Message types used in the conversation:
# HumanMessage  — the user's input
# SystemMessage — a standing instruction that sets the model's persona/behavior
# ToolMessage   — the result returned by a tool after the model calls it
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper to print every message in the conversation with role labels (Human/AI/Tool)
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


In [ ]:
# Define tools that the model can call during a conversation.
# The @tool decorator registers the function with LangChain so the model
# can discover it and decide when to call it based on its docstring description.

@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """
    # Hard-coded lookup table: artist name (lowercase) → venue description
    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    # Normalize to lowercase and strip whitespace before lookup
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    # Hard-coded route table mapping (origin, destination) tuples to travel time strings
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    # Build the lookup key by normalizing both city names
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")


In [ ]:
# Register the tools and build a lookup registry (name → tool function)
tools = [lookup_tour_stop, estimate_drive_time]
tools_registry = { t.name: t for t in tools }

# Attach the tools to the model using bind_tools().
# After this, the model knows about the tools and can request to call them
# by returning a structured tool_calls field in its response.
openai_default_model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(tools)

# Start the conversation with a system instruction and a user question
prev_messages = [
    SystemMessage("You are a practical live-music concierge. Use tools when they help you give a precise answer."),
    HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time."),
]

# This loop implements the manual "agent loop" — the core of how tool-calling works:
#   1. Send the current conversation to the model
#   2. If the model returns tool calls, execute them and append the results
#   3. Send everything back to the model for the next reasoning step
#   4. Repeat until the model returns a plain text answer (no more tool calls)
MAX_ITERATIONS = 100  # Safety limit to prevent infinite loops
finished_successfully = False

for i in range(MAX_ITERATIONS):
    # Call the model with the current conversation history
    reply = openai_default_model.invoke(prev_messages)
    prev_messages.append(reply)  # Always add the reply to the conversation

    # If the model returned no tool calls, it's done — we have a final answer
    if not reply.tool_calls:
        finished_successfully = True
        break

    # Otherwise, execute each tool the model requested
    for tool_call in reply.tool_calls:
        tool_call_id = tool_call["id"]       # Unique ID that links this call to its result
        tool_call_name = tool_call["name"]   # Which tool to call (e.g., "lookup_tour_stop")
        tool_call_args = tool_call["args"]   # Arguments the model chose to pass

        # Look up the tool function in our registry and call it with the provided arguments
        result = tools_registry[tool_call_name].invoke(tool_call_args)

        # Wrap the tool result in a ToolMessage and append it to the conversation.
        # The tool_call_id links this result back to the specific tool call that requested it.
        prev_messages.append(ToolMessage(str(result), tool_call_id=tool_call_id))


if not finished_successfully:
    raise RuntimeError(f"Could not finish the interaction within {MAX_ITERATIONS} iterations.")


In [ ]:
# Print the full conversation including system prompt, user question,
# the tool calls the model made, the tool results, and the final AI answer.
print_conversation(prev_messages)
